# Python-Based Inventory & Order Management Application

## 1. Custom exceptions

In [1]:
import json
from datetime import datetime

# Custom errors: they give clear messages when something goes wrong
class InventoryError(Exception):
    """Base class for all errors in this app."""

class InvalidInputError(InventoryError):
    """Raised when a value is wrong (negative price, empty name, etc.)."""

class RecordNotFoundError(InventoryError):
    """Raised when a product, customer or order does not exist."""

class OutOfStockError(InventoryError):
    """Raised when an order asks for more items than we have."""

## 2. Classes: Product, Customer, Order

In [2]:
class Product:
    """One item we sell."""

    def __init__(self, product_id, name, price, quantity):
        if not str(name).strip():
            raise InvalidInputError("Product name cannot be empty.")
        if price < 0:
            raise InvalidInputError("Price cannot be negative.")
        if quantity < 0:
            raise InvalidInputError("Quantity cannot be negative.")
        self.product_id = product_id
        self.name = name.strip()
        self.price = price
        self.quantity = quantity

    def to_dict(self):
        return {"product_id": self.product_id, "name": self.name,
                "price": self.price, "quantity": self.quantity}

    @classmethod
    def from_dict(cls, d):
        return cls(d["product_id"], d["name"], d["price"], d["quantity"])

    def __str__(self):
        return f"[{self.product_id}] {self.name} | Rs.{self.price:.2f} | Stock: {self.quantity}"

In [3]:
class Customer:
    """One person who places orders."""

    def __init__(self, customer_id, name, email):
        if not str(name).strip():
            raise InvalidInputError("Customer name cannot be empty.")
        if "@" not in str(email):
            raise InvalidInputError(f"Invalid email: {email}")
        self.customer_id = customer_id
        self.name = name.strip()
        self.email = email.strip()

    def to_dict(self):
        return {"customer_id": self.customer_id, "name": self.name, "email": self.email}

    @classmethod
    def from_dict(cls, d):
        return cls(d["customer_id"], d["name"], d["email"])

    def __str__(self):
        return f"[{self.customer_id}] {self.name} <{self.email}>"

In [4]:
class Order:
    """One purchase: a customer, a list of items, a date and a total."""

    def __init__(self, order_id, customer_id, items, date=None):
        # items is a list of dicts: {"product_id", "name", "price", "quantity"}
        if not items:
            raise InvalidInputError("An order needs at least one item.")
        self.order_id = order_id
        self.customer_id = customer_id
        self.items = items
        self.date = date or datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    @property
    def total(self):
        return sum(i["price"] * i["quantity"] for i in self.items)

    def to_dict(self):
        return {"order_id": self.order_id, "customer_id": self.customer_id,
                "items": self.items, "date": self.date}

    @classmethod
    def from_dict(cls, d):
        return cls(d["order_id"], d["customer_id"], d["items"], d["date"])

    def __str__(self):
        lines = [f"Order #{self.order_id} | Customer {self.customer_id} | {self.date}"]
        for i in self.items:
            lines.append(f"   {i['name']} x{i['quantity']} @ Rs.{i['price']:.2f}")
        lines.append(f"   TOTAL: Rs.{self.total:.2f}")
        return "\n".join(lines)

### Testing the classes

In [5]:
p = Product(1, "Laptop", 55000, 10)
c = Customer(1, "Asha", "asha@example.com")
o = Order(1, 1, [{"product_id": 1, "name": "Laptop", "price": 55000, "quantity": 2}])

print(p)
print(c)
print(o)

# Test the validation: this should be caught, not crash
try:
    Product(2, "Mouse", -500, 5)
except InvalidInputError as e:
    print("Caught error:", e)

[1] Laptop | Rs.55000.00 | Stock: 10
[1] Asha <asha@example.com>
Order #1 | Customer 1 | 2026-10-04 02:42:03
   Laptop x2 @ Rs.55000.00
   TOTAL: Rs.110000.00
Caught error: Price cannot be negative.


## 3. Managers: add, update, search, delete, stock tracking

In [6]:
class RecordManager:
    """Stores products and customers. Handles add / get / update / delete / search."""

    def __init__(self):
        self.products = {}      # product_id -> Product
        self.customers = {}     # customer_id -> Customer
        self.orders = {}        # order_id -> Order
        self.next_product_id = 1
        self.next_customer_id = 1
        self.next_order_id = 1

    # ---------- Products ----------
    def add_product(self, name, price, quantity):
        if not isinstance(price, (int, float)) or not isinstance(quantity, int):
            raise InvalidInputError("Price must be a number and quantity a whole number.")
        product = Product(self.next_product_id, name, price, quantity)
        self.products[product.product_id] = product
        self.next_product_id += 1
        return product

    def get_product(self, product_id):
        if product_id not in self.products:
            raise RecordNotFoundError(f"Product {product_id} not found.")
        return self.products[product_id]

    def update_product(self, product_id, name=None, price=None, quantity=None):
        old = self.get_product(product_id)
        # Building a new Product re-runs all the validation checks
        new = Product(product_id,
                      old.name if name is None else name,
                      old.price if price is None else price,
                      old.quantity if quantity is None else quantity)
        self.products[product_id] = new
        return new

    def delete_product(self, product_id):
        self.get_product(product_id)      # raises an error if it does not exist
        del self.products[product_id]

    def search_products(self, keyword):
        keyword = keyword.lower()
        return [p for p in self.products.values() if keyword in p.name.lower()]

    # ---------- Customers ----------
    def add_customer(self, name, email):
        customer = Customer(self.next_customer_id, name, email)
        self.customers[customer.customer_id] = customer
        self.next_customer_id += 1
        return customer

    def get_customer(self, customer_id):
        if customer_id not in self.customers:
            raise RecordNotFoundError(f"Customer {customer_id} not found.")
        return self.customers[customer_id]

    def update_customer(self, customer_id, name=None, email=None):
        old = self.get_customer(customer_id)
        new = Customer(customer_id,
                       old.name if name is None else name,
                       old.email if email is None else email)
        self.customers[customer_id] = new
        return new

    def delete_customer(self, customer_id):
        self.get_customer(customer_id)
        del self.customers[customer_id]

    def search_customers(self, keyword):
        keyword = keyword.lower()
        return [c for c in self.customers.values()
                if keyword in c.name.lower() or keyword in c.email.lower()]

In [7]:
m = RecordManager()
m.add_product("Laptop", 55000, 10)
m.add_product("Mouse", 500, 50)
m.add_product("Keyboard", 1500, 3)
m.add_customer("Asha", "asha@example.com")
m.add_customer("Ravi", "ravi@example.com")

print("All products:")
for p in m.products.values():
    print("  ", p)

m.update_product(2, price=550)
print("After update:", m.get_product(2))

print("Search 'key':")
for p in m.search_products("key"):
    print("  ", p)

m.delete_product(3)
print("Products left:", len(m.products))

try:
    m.get_product(3)
except RecordNotFoundError as e:
    print("Caught error:", e)

All products:
   [1] Laptop | Rs.55000.00 | Stock: 10
   [2] Mouse | Rs.500.00 | Stock: 50
   [3] Keyboard | Rs.1500.00 | Stock: 3
After update: [2] Mouse | Rs.550.00 | Stock: 50
Search 'key':
   [3] Keyboard | Rs.1500.00 | Stock: 3
Products left: 2
Caught error: Product 3 not found.


In [8]:
class InventoryManager(RecordManager):
    """Everything RecordManager does, plus placing orders and tracking stock."""

    def place_order(self, customer_id, items):
        # items is a list of (product_id, quantity) pairs, e.g. [(1, 2), (2, 5)]
        self.get_customer(customer_id)
        if not items:
            raise InvalidInputError("An order needs at least one item.")

        # Pass 1: validate everything and add up quantities per product
        needed = {}
        for product_id, qty in items:
            if not isinstance(qty, int) or qty <= 0:
                raise InvalidInputError("Quantity must be a positive whole number.")
            needed[product_id] = needed.get(product_id, 0) + qty

        for product_id, qty in needed.items():
            product = self.get_product(product_id)
            if product.quantity < qty:
                raise OutOfStockError(
                    f"Not enough stock for '{product.name}': "
                    f"requested {qty}, available {product.quantity}.")

        # Pass 2: everything is valid, so reduce the stock and create the order
        order_items = []
        for product_id, qty in needed.items():
            product = self.products[product_id]
            product.quantity -= qty
            order_items.append({"product_id": product_id, "name": product.name,
                                "price": product.price, "quantity": qty})

        order = Order(self.next_order_id, customer_id, order_items)
        self.orders[order.order_id] = order
        self.next_order_id += 1
        return order

    def low_stock_products(self, threshold=5):
        return [p for p in self.products.values() if p.quantity <= threshold]

In [9]:
inv = InventoryManager()
inv.add_product("Laptop", 55000, 10)
inv.add_product("Mouse", 500, 50)
inv.add_product("Keyboard", 1500, 3)
inv.add_customer("Asha", "asha@example.com")

print("Laptop stock before:", inv.get_product(1).quantity)
order = inv.place_order(1, [(1, 2), (2, 5)])
print(order)
print("Laptop stock after:", inv.get_product(1).quantity)
print("Mouse stock after:", inv.get_product(2).quantity)

Laptop stock before: 10
Order #1 | Customer 1 | 2026-10-04 02:42:04
   Laptop x2 @ Rs.55000.00
   Mouse x5 @ Rs.500.00
   TOTAL: Rs.112500.00
Laptop stock after: 8
Mouse stock after: 45


In [10]:
bad_orders = [
    (1, [(1, 1), (3, 10)]),   # asks for 10 keyboards, only 3 exist
    (99, [(1, 1)]),           # customer does not exist
    (1, [(99, 1)]),           # product does not exist
    (1, [(1, 0)]),            # quantity of zero
]

for customer_id, items in bad_orders:
    try:
        inv.place_order(customer_id, items)
    except InventoryError as e:
        print(type(e).__name__, "-", e)

print("Laptop stock is still:", inv.get_product(1).quantity)

OutOfStockError - Not enough stock for 'Keyboard': requested 10, available 3.
RecordNotFoundError - Customer 99 not found.
RecordNotFoundError - Product 99 not found.
InvalidInputError - Quantity must be a positive whole number.
Laptop stock is still: 8


In [11]:
print("Low stock items:")
for p in inv.low_stock_products(threshold=5):
    print("  ", p)

print("Total orders placed:", len(inv.orders))

Low stock items:
   [3] Keyboard | Rs.1500.00 | Stock: 3
Total orders placed: 1


## 4. File persistence (JSON)

In [12]:
import os
import csv

class PersistentInventoryManager(InventoryManager):
    """InventoryManager plus the ability to save to and load from a JSON file."""

    def save(self, filename="inventory_data.json"):
        data = {
            "products": [p.to_dict() for p in self.products.values()],
            "customers": [c.to_dict() for c in self.customers.values()],
            "orders": [o.to_dict() for o in self.orders.values()],
            "next_product_id": self.next_product_id,
            "next_customer_id": self.next_customer_id,
            "next_order_id": self.next_order_id,
        }
        with open(filename, "w") as f:
            json.dump(data, f, indent=2)

    def load(self, filename="inventory_data.json"):
        if not os.path.exists(filename):
            raise RecordNotFoundError(f"File '{filename}' not found.")
        try:
            with open(filename, "r") as f:
                data = json.load(f)
        except json.JSONDecodeError:
            raise InvalidInputError(f"File '{filename}' is damaged or not valid JSON.")

        self.products = {d["product_id"]: Product.from_dict(d) for d in data["products"]}
        self.customers = {d["customer_id"]: Customer.from_dict(d) for d in data["customers"]}
        self.orders = {d["order_id"]: Order.from_dict(d) for d in data["orders"]}
        self.next_product_id = data["next_product_id"]
        self.next_customer_id = data["next_customer_id"]
        self.next_order_id = data["next_order_id"]

In [13]:
pm = PersistentInventoryManager()
pm.add_product("Laptop", 55000, 10)
pm.add_product("Mouse", 500, 50)
pm.add_customer("Asha", "asha@example.com")
pm.place_order(1, [(1, 2)])
pm.save()
print("Saved to inventory_data.json")

# A brand new, empty manager that loads the file
fresh = PersistentInventoryManager()
fresh.load()
print("Products loaded:", len(fresh.products))
print("Laptop stock:", fresh.get_product(1).quantity)
print("Orders loaded:", len(fresh.orders))
print("Next order id:", fresh.next_order_id)

try:
    fresh.load("nofile.json")
except InventoryError as e:
    print(type(e).__name__, "-", e)

Saved to inventory_data.json
Products loaded: 2
Laptop stock: 8
Orders loaded: 1
Next order id: 2
RecordNotFoundError - File 'nofile.json' not found.


## 5. Reports and transaction records

In [14]:
class InventoryApp(PersistentInventoryManager):
    """The final application: everything above plus reports and transaction records."""

    def inventory_report(self):
        print("INVENTORY REPORT")
        print(f"{'ID':<4}{'Name':<12}{'Price':>10}{'Stock':>7}{'Value':>12}")
        total = 0
        for p in self.products.values():
            value = p.price * p.quantity
            total += value
            print(f"{p.product_id:<4}{p.name:<12}{p.price:>10.2f}{p.quantity:>7}{value:>12.2f}")
        print(f"Total stock value: Rs.{total:.2f}")

    def order_summary(self, order_id):
        if order_id not in self.orders:
            raise RecordNotFoundError(f"Order {order_id} not found.")
        order = self.orders[order_id]
        customer = self.customers.get(order.customer_id)
        name = customer.name if customer else "Unknown (deleted)"
        print(f"Customer: {name}")
        print(order)

    def export_transactions(self, filename="transactions.csv"):
        rows = 0
        with open(filename, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["order_id", "date", "customer", "product",
                             "quantity", "unit_price", "line_total"])
            for order in self.orders.values():
                customer = self.customers.get(order.customer_id)
                name = customer.name if customer else "Unknown"
                for item in order.items:
                    writer.writerow([order.order_id, order.date, name, item["name"],
                                     item["quantity"], item["price"],
                                     item["price"] * item["quantity"]])
                    rows += 1
        return rows

In [15]:
app = InventoryApp()
app.load()                                    # reads the file from Step 26
app.add_product("Keyboard", 1500, 3)
app.add_customer("Ravi", "ravi@example.com")
app.place_order(2, [(2, 5), (3, 1)])

app.inventory_report()
print()
app.order_summary(2)

INVENTORY REPORT
ID  Name             Price  Stock       Value
1   Laptop        55000.00      8   440000.00
2   Mouse           500.00     45    22500.00
3   Keyboard       1500.00      2     3000.00
Total stock value: Rs.465500.00

Customer: Ravi
Order #2 | Customer 2 | 2026-10-04 02:42:04
   Mouse x5 @ Rs.500.00
   Keyboard x1 @ Rs.1500.00
   TOTAL: Rs.4000.00


In [16]:
count = app.export_transactions()
print("Rows written:", count)
print()
with open("transactions.csv") as f:
    print(f.read())

app.save()    # save the updated data too

Rows written: 3

order_id,date,customer,product,quantity,unit_price,line_total
1,2026-10-04 02:42:04,Asha,Laptop,2,55000,110000
2,2026-10-04 02:42:04,Ravi,Mouse,5,500,2500
2,2026-10-04 02:42:04,Ravi,Keyboard,1,1500,1500



## 6. Full demonstration

In [17]:
demo = InventoryApp()

print("=== 1. ADD PRODUCTS AND CUSTOMERS ===")
demo.add_product("Laptop", 55000, 10)
demo.add_product("Mouse", 500, 50)
demo.add_product("Keyboard", 1500, 3)
demo.add_product("Monitor", 12000, 6)
demo.add_customer("Asha", "asha@example.com")
demo.add_customer("Ravi", "ravi@example.com")
demo.inventory_report()

print("\n=== 2. UPDATE RECORDS ===")
demo.update_product(2, price=550)
demo.update_customer(2, email="ravi.new@example.com")
print(demo.get_product(2))
print(demo.get_customer(2))

print("\n=== 3. SEARCH ===")
print("Products matching 'o':", [p.name for p in demo.search_products("o")])
print("Customers matching 'asha':", [c.name for c in demo.search_customers("asha")])

print("\n=== 4. PLACE ORDERS (stock updates automatically) ===")
demo.place_order(1, [(1, 2), (2, 5)])
demo.place_order(2, [(3, 1), (4, 2)])
demo.order_summary(1)
print()
demo.order_summary(2)

print("\n=== 5. STOCK AFTER ORDERS ===")
demo.inventory_report()
print("Low stock (5 or fewer):", [p.name for p in demo.low_stock_products(5)])

print("\n=== 6. DELETE A RECORD ===")
demo.delete_product(4)
print("Products left:", [p.name for p in demo.products.values()])

print("\n=== 7. ERROR HANDLING ===")
error_tests = [
    ("Negative price",   lambda: demo.add_product("Cable", -10, 5)),
    ("Empty name",       lambda: demo.add_product("   ", 10, 5)),
    ("Price not number", lambda: demo.add_product("Cable", "abc", 5)),
    ("Bad email",        lambda: demo.add_customer("Sam", "samexample.com")),
    ("Not enough stock", lambda: demo.place_order(1, [(3, 99)])),
    ("Unknown customer", lambda: demo.place_order(42, [(1, 1)])),
    ("Deleted product",  lambda: demo.get_product(4)),
    ("Missing file",     lambda: demo.load("missing.json")),
]
for label, action in error_tests:
    try:
        action()
        print(f"{label}: NO ERROR (unexpected)")
    except InventoryError as e:
        print(f"{label}: {type(e).__name__} - {e}")

print("\n=== 8. SAVE, LOAD AND EXPORT ===")
demo.save("demo_inventory.json")
reloaded = InventoryApp()
reloaded.load("demo_inventory.json")
print("Reloaded products:", len(reloaded.products),
      "| customers:", len(reloaded.customers),
      "| orders:", len(reloaded.orders))
print("Transaction rows exported:", demo.export_transactions("demo_transactions.csv"))

=== 1. ADD PRODUCTS AND CUSTOMERS ===
INVENTORY REPORT
ID  Name             Price  Stock       Value
1   Laptop        55000.00     10   550000.00
2   Mouse           500.00     50    25000.00
3   Keyboard       1500.00      3     4500.00
4   Monitor       12000.00      6    72000.00
Total stock value: Rs.651500.00

=== 2. UPDATE RECORDS ===
[2] Mouse | Rs.550.00 | Stock: 50
[2] Ravi <ravi.new@example.com>

=== 3. SEARCH ===
Products matching 'o': ['Laptop', 'Mouse', 'Keyboard', 'Monitor']
Customers matching 'asha': ['Asha']

=== 4. PLACE ORDERS (stock updates automatically) ===
Customer: Asha
Order #1 | Customer 1 | 2026-10-04 02:42:04
   Laptop x2 @ Rs.55000.00
   Mouse x5 @ Rs.550.00
   TOTAL: Rs.112750.00

Customer: Ravi
Order #2 | Customer 2 | 2026-10-04 02:42:04
   Keyboard x1 @ Rs.1500.00
   Monitor x2 @ Rs.12000.00
   TOTAL: Rs.25500.00

=== 5. STOCK AFTER ORDERS ===
INVENTORY REPORT
ID  Name             Price  Stock       Value
1   Laptop        55000.00      8   440000.00
2  